# Lahore UC: NDVI and Nightlights (GEE → Local)

This notebook:

- Authenticates and initializes Google Earth Engine (GEE).
- Computes UC-level NDVI (Sentinel-2 SR, B8/B4) for a given date range and exports:
  - UC_NDVI.geojson
  - UC_NDVI.csv
- Computes UC-level VIIRS DNB nightlights (avg_rad) and exports:
  - UC_Nightlights.geojson
  - UC_Nightlights.csv
- Visualizes NDVI (and optionally Nightlights) as Folium choropleths.

Tips:
- Make sure you’ve installed `earthengine-api` and `geemap` in this kernel.
- First run the authentication cell, then the analysis/export cells.

In [2]:
import ee

# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize()


In [3]:
# !pip install geemap

import ee
import geemap
import pandas as pd

ee.Initialize()

# Load UC boundaries
uc_asset = "projects/ee-ahmedabclr35/assets/Lahore_union_Council_Boundries"
ucs = ee.FeatureCollection(uc_asset)

# Define time range
start = '2024-01-01'
end   = '2024-12-31'

# Sentinel-2 Surface Reflectance, filter + select only B4, B8
s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
        .filterDate(start, end)
        .filterBounds(ucs)
        .select(['B4', 'B8'])  # ✅ keep only bands we need
        .median())


ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Reduce to UC level
uc_stats = ndvi.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.mean(),
    scale=10
)

# --- Option 1: Export locally as GeoJSON ---
geemap.ee_export_vector(uc_stats, filename="UC_NDVI.geojson")

# --- Option 2: Export locally as CSV ---
df = geemap.ee_to_pandas(uc_stats)
df.to_csv("UC_NDVI.csv", index=False)

print("Exported UC stats locally!")


Generating URL ...
Please wait ...
Data downloaded to d:\LUMS\Senior Fall 25\SPROJ - Dr Tahir\SPROJ\notebooks\UC_NDVI.geojson


AttributeError: module 'geemap' has no attribute 'ee_to_pandas'

## 2) Sentinel-2 NDVI per UC (export)
Filters S2 SR to your UC layer and date range, computes NDVI = (B8 - B4) / (B8 + B4), reduces mean per UC, and exports GeoJSON + CSV locally.

In [10]:
# --- VIIRS Nightlights by UC (GeoJSON + CSV export) ---

import ee, geemap, pandas as pd
from datetime import datetime

# Reuse existing variables if present; otherwise set defaults

uc_asset = "projects/ee-ahmedabclr35/assets/Lahore_union_Council_Boundries"
ucs = ee.FeatureCollection(uc_asset)

try:
    _ = start; _ = end
except NameError:
    start = '2025-01-01'
    end   = '2025-08-31'

# Prefer nightly lights stable, cloud-free product; fall back if empty
viirs_primary_id = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"   # stray-light corrected
viirs_fallback_id = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG"    # fallback if above empty

def viirs_collection(col_id):
    return (ee.ImageCollection(col_id)
            .filterDate(start, end)
            .filterBounds(ucs)
            .select(["avg_rad"]))

col = viirs_collection(viirs_primary_id)
count = col.size().getInfo()
if count == 0:
    col = viirs_collection(viirs_fallback_id)
    print("Primary VIIRS collection empty for range; using fallback VCMCFG.")

# Aggregate to mean nightlights for the period
viirs_mean = col.mean().rename("avg_rad")

# Reduce to UC-level features (mean radiance)
nl_stats = viirs_mean.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.mean(),
    scale=500
)

# Export locally
geemap.ee_export_vector(nl_stats, filename="UC_Nightlights.geojson")
df_nl = geemap.ee_to_pandas(nl_stats)
df_nl.to_csv("UC_Nightlights.csv", index=False)
print("Exported UC Nightlights → UC_Nightlights.geojson, UC_Nightlights.csv")

Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/UC_Nightlights.geojson
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/UC_Nightlights.geojson


AttributeError: module 'geemap' has no attribute 'ee_to_pandas'

## 3) VIIRS Nightlights per UC (export)
Computes mean monthly VIIRS DNB radiance (avg_rad) over the date range, reduces mean per UC, and exports GeoJSON + CSV. Falls back from VCMSLCFG to VCMCFG if the primary is empty.

In [11]:
# Get all features into Python dict
features = nl_stats.getInfo()['features']

# Flatten into list of dicts
rows = [f['properties'] for f in features]

# Convert to DataFrame
df_nl = pd.DataFrame(rows)

# Save locally
df_nl.to_csv("UC_Nightlights.csv", index=False)
print("Saved UC Nightlights locally!")


Saved UC Nightlights locally!


In [5]:
import ee, geemap, pandas as pd
ee.Initialize()

# Get all features into Python dict
features = uc_stats.getInfo()['features']

# Flatten into list of dicts
rows = [f['properties'] for f in features]

# Convert to DataFrame
df = pd.DataFrame(rows)

# Save locally
df.to_csv("UC_NDVI.csv", index=False)
print("Saved UC NDVI locally!")


Saved UC NDVI locally!


## 4) Alternate: pull features to client and write CSV
If you prefer pure pandas without geemap’s convenience exporters, you can pull `uc_stats` to the client and write a CSV directly. This can be slower for large collections.

In [8]:
import geopandas as gpd
import pandas as pd
import folium

# Load UC boundaries (GeoJSON)
uc_gdf = gpd.read_file("../data/Union_Councils.geojson")

# Load stats (CSV from GEE export)
stats_df = pd.read_csv("UC_NDVI.csv")

merged = uc_gdf.merge(stats_df, on="Name_UC_N")  

# Initialize folium map
m = folium.Map(location=[31.5204, 74.3587], zoom_start=11, tiles="cartodbpositron")

# Choropleth map (e.g. NDVI mean)
folium.Choropleth(
    geo_data=merged,
    data=merged,
    columns=["Name_UC_N", "mean"],   # UC name and the value to color
    key_on="feature.properties.Name_UC_N",
    fill_color="YlGn",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="NDVI Mean"
).add_to(m)

# Add tooltips (hover labels)
folium.GeoJson(
    merged,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    tooltip=folium.GeoJsonTooltip(
        fields=["Name_UC_N", "mean", "P_D_16_x"],  # what to show on hover
        aliases=["UC Name:", "NDVI Mean:", "Pop Density 2016:"],
        localize=True
    )
).add_to(m)

# Save to HTML
m.save("UC_NDVI_heatmap.html")

## 5) Folium visualization for NDVI
Join the UC GeoJSON with exported NDVI CSV and render a choropleth; customize fields and legend as needed.

In [12]:
# Merge UC boundaries with nightlights stats
nl_stats_df = pd.DataFrame(rows)
merged_nl = uc_gdf.merge(nl_stats_df, on="Name_UC_N")

# Initialize folium map for nightlights
m_nl = folium.Map(location=[31.5204, 74.3587], zoom_start=11, tiles="cartodbpositron")

# Choropleth map for VIIRS nightlights (mean radiance)
folium.Choropleth(
    geo_data=merged_nl,
    data=merged_nl,
    columns=["Name_UC_N", "mean"],   # UC name and the value to color
    key_on="feature.properties.Name_UC_N",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="VIIRS Nightlights Mean"
).add_to(m_nl)

# Add tooltips (hover labels)
folium.GeoJson(
    merged_nl,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    tooltip=folium.GeoJsonTooltip(
        fields=["Name_UC_N", "mean", "P_D_16_x"],
        aliases=["UC Name:", "Nightlights Mean:", "Pop Density 2016:"],
        localize=True
    )
).add_to(m_nl)

# Save to HTML
m_nl.save("UC_Nightlights_heatmap.html")

#building footprint + lst + pollution map

## 6) (Optional) Folium visualization for Nightlights
You can mirror the NDVI map using `UC_Nightlights.csv` by merging on the same UC name field and mapping the `mean` radiance column. Save as, e.g., `UC_Nightlights_heatmap.html`.